Import libraries and define the Dataset A path

In [13]:

# DATASET A AUDIT

# Purpose:
# This notebook is the first step of the IBY project.
# We will inspect Dataset A to understand:
#   1. How the data is organized
#   2. How sessions and chunks are structured
#   3. What information is contained in the event logs
#   4. What the ground-truth files look like
#
# Dataset A contains ground truth, so it will later be used
# to develop and validate our process-segmentation approach.


import json
from pathlib import Path
from collections import Counter

# Path to the combined Dataset A
dataset_a = Path("dataset_A") / "dataset_a_combined"

print("Dataset A path:", dataset_a)
print("Path exists:", dataset_a.exists())

Dataset A path: dataset_A\dataset_a_combined
Path exists: True


Find all sessions

In [14]:

# IDENTIFY SESSIONS

# Each session represents one recording period.
# A session can contain multiple chunks, so we must NOT
# treat every chunk as an independent session.


sessions = sorted([
    p for p in dataset_a.iterdir()
    if p.is_dir() and p.name.startswith("ses_")
])

print("Number of sessions:", len(sessions))

print("\nFirst 5 sessions:")
for session in sessions[:5]:
    print(" -", session.name)

Number of sessions: 63

First 5 sessions:
 - ses_20260630-121953-LAPTOP-R36BQBTE
 - ses_20260630-124826-CHAITANYA0BCF
 - ses_20260630-125757-LAPTOP-R36BQBTE
 - ses_20260630-131729-CHAITANYA0BCF
 - ses_20260630-132737-LAPTOP-R36BQBTE


Inspect the first session

In [15]:

# INSPECT ONE SESSION

# We begin with one session rather than loading the entire
# dataset. This lets us understand the data structure before
# designing the segmentation algorithm.


session = sessions[0]

print("Selected session:")
print(session.name)

print("\nContents of the session:")

for item in sorted(session.iterdir()):
    print(" -", item.name)

Selected session:
ses_20260630-121953-LAPTOP-R36BQBTE

Contents of the session:
 - chunk_1200
 - chunk_1230
 - chunk_20260630-1200-LAPTOP-R36BQBTE
 - chunk_20260630-1230-LAPTOP-R36BQBTE
 - gt.jsonl
 - gt_manifest.json


Find the event logs

In [16]:

# FIND EVENT LOGS

# A single session may be divided into multiple chunks.
# Each chunk contains an events.jsonl file.
#
# Therefore, we recursively search inside the session instead
# of assuming that there is only one events.jsonl file.


event_files = sorted(session.rglob("events.jsonl"))

print("Number of event-log files (chunks):", len(event_files))

print("\nEvent files:")

for file in event_files:
    print(" -", file)

Number of event-log files (chunks): 2

Event files:
 - dataset_A\dataset_a_combined\ses_20260630-121953-LAPTOP-R36BQBTE\chunk_20260630-1200-LAPTOP-R36BQBTE\events.jsonl
 - dataset_A\dataset_a_combined\ses_20260630-121953-LAPTOP-R36BQBTE\chunk_20260630-1230-LAPTOP-R36BQBTE\events.jsonl


Inspect the actual events

In [17]:

# INSPECT RAW EVENTS

# We read only the first 10 events.
#
# The purpose is NOT to analyze the whole dataset yet.
# We first need to discover the actual schema of the events:
# timestamps, event types, applications, mouse/keyboard data,
# screenshots, etc.
#
# We should not assume the field names before inspecting them.


event_file = event_files[0]

print("Inspecting:")
print(event_file)
print("\nFirst 10 events:\n")

with open(event_file, "r", encoding="utf-8") as f:

    for i in range(10):

        line = f.readline()

        if not line:
            break

        event = json.loads(line)

        print(f"--- Event {i + 1} ---")
        print(json.dumps(event, indent=2, ensure_ascii=False))

Inspecting:
dataset_A\dataset_a_combined\ses_20260630-121953-LAPTOP-R36BQBTE\chunk_20260630-1200-LAPTOP-R36BQBTE\events.jsonl

First 10 events:

--- Event 1 ---
{
  "schema_version": "1.0.0",
  "event_id": "evt_f29ff271-1691-4070-9b8f-02aeead78ff2",
  "session_id": "ses_20260630-121953-LAPTOP-R36BQBTE",
  "timestamp_ms": 1782821993821,
  "timestamp_iso": "2026-06-30T12:19:53.821Z",
  "layer": "SYSTEM",
  "event_type": "session_start",
  "source": {
    "agent_version": "1.1.1",
    "machine_id": "LAPTOP-R36BQBTE",
    "os": "Windows Windows_NT",
    "username_hash": "sha256:8c2ea5ba76c042f4"
  },
  "context": {
    "active_app": null,
    "active_monitor": null,
    "active_browser_tab": null,
    "visible_windows": null,
    "open_apps": null
  },
  "correlation": {
    "triggered_by": null,
    "correlated_events": [],
    "sequence_number": 0,
    "chunk_id": "chunk_20260630-1200-LAPTOP-R36BQBTE",
    "ms_since_last_event": null
  },
  "payload": {
    "agent_version": "1.1.1",
    

Inspect the ground truth

In [18]:

# INSPECT GROUND TRUTH

# Dataset A provides ground truth through gt.jsonl.
#
# This file tells us where business processes occurred.
# We will eventually use this information to evaluate how well
# our segmentation method recovers individual units of work.


gt_file = session / "gt.jsonl"

print("Ground-truth file:")
print(gt_file)

print("\nFirst 10 ground-truth records:\n")

with open(gt_file, "r", encoding="utf-8") as f:

    for i in range(10):

        line = f.readline()

        if not line:
            break

        gt = json.loads(line)

        print(f"--- Ground Truth {i + 1} ---")
        print(json.dumps(gt, indent=2, ensure_ascii=False))

Ground-truth file:
dataset_A\dataset_a_combined\ses_20260630-121953-LAPTOP-R36BQBTE\gt.jsonl

First 10 ground-truth records:

--- Ground Truth 1 ---
{
  "ts_utc": "2026-06-30T12:20:25.140811+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "run_config",
  "seed": 645230,
  "dwell_scale": 1.4,
  "n_procs": 9,
  "run_date": "2026-06-30",
  "operator": "田中 健一",
  "operator_dept": "人事部",
  "machine_id": "NB-M1-05",
  "noise_rate": 0.2,
  "noise_blocks": 2,
  "chunk_split": "time:420",
  "continuation_family": "B",
  "continuation_dwell": 2.0,
  "split_min_gap": 6,
  "duration": "medium",
  "domain_mix": {},
  "transition": "random",
  "operator_type": "expert",
  "tasks_per_proc": [
    3,
    5
  ],
  "selected_procs": [
    "E",
    "C",
    "A",
    "B",
    "H",
    "J",
    "G",
    "O",
    "M"
  ],
  "split_procs": [
    "B"
  ],
  "session_notes": "田中 健一 — 9 procs — 2026-06-30 — seed 645230"
}
--- Ground Truth 2 ---
{
  "ts_utc": "2026-06-30T12:21:12.794418+00:00",
  "run

Count event types

In [19]:

# INSPECT EVENT SCHEMA

# Before counting event types, we first inspect which keys
# are actually present in the event records.


event_keys = Counter()

with open(event_file, "r", encoding="utf-8") as f:

    for line in f:

        event = json.loads(line)

        event_keys.update(event.keys())

print("Event fields found:")
print()

for key, count in event_keys.most_common():
    print(f"{key}: {count}")

Event fields found:

schema_version: 1116
event_id: 1116
session_id: 1116
timestamp_ms: 1116
timestamp_iso: 1116
layer: 1116
event_type: 1116
source: 1116
context: 1116
correlation: 1116
payload: 1116
metadata: 1116
extensions: 1116


Basic event count

In [20]:

# COUNT EVENTS IN THE SELECTED CHUNK


event_count = 0

with open(event_file, "r", encoding="utf-8") as f:

    for line in f:

        if line.strip():
            event_count += 1

print("Events in selected chunk:", event_count)

Events in selected chunk: 1116


ground-truth record type inventory

In [21]:
gt_event_types = Counter()

with open(gt_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        record = json.loads(line)
        record_type = record.get("event", "UNKNOWN")

        gt_event_types[record_type] += 1

print("Ground-truth record types:")
print("-" * 50)

for record_type, count in gt_event_types.most_common():
    print(f"{record_type:<35} {count:>8}")

print("-" * 50)
print("Total ground-truth records:", sum(gt_event_types.values()))

Ground-truth record types:
--------------------------------------------------
clipboard_paste                           50
task_started                              32
clipboard_copy                            32
process_started                           31
process_switched_out                      26
run_config                                 1
process_suspended                          1
process_resumed                            1
session_ended                              1
--------------------------------------------------
Total ground-truth records: 175


process-related ground-truth inspection

In [22]:

# CELL 10 — INSPECT PROCESS-RELATED GROUND TRUTH


process_events = []

with open(gt_file, "r", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        if record.get("event") in {
            "process_started",
            "process_switched_out",
            "process_suspended",
            "process_resumed"
        }:
            process_events.append(record)

print("Number of process-related records:", len(process_events))

print("\nFirst 10 process-related records:\n")

for i, record in enumerate(process_events[:10], start=1):

    print(f"--- Process Record {i} ---")
    print(json.dumps(record, indent=2, ensure_ascii=False))

Number of process-related records: 59

First 10 process-related records:

--- Process Record 1 ---
{
  "ts_utc": "2026-06-30T12:21:12.794418+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "process_started",
  "current_process": "H",
  "process_code": "H",
  "process_name": "銀行勘定照合",
  "case_id": "BR-175009-001"
}
--- Process Record 2 ---
{
  "ts_utc": "2026-06-30T12:21:41.332221+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "process_switched_out",
  "current_process": "H",
  "from": "H",
  "to": "C"
}
--- Process Record 3 ---
{
  "ts_utc": "2026-06-30T12:21:41.335282+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "process_started",
  "current_process": "C",
  "process_code": "C",
  "process_name": "育児・産休申請確認",
  "case_id": "LA-175009-001"
}
--- Process Record 4 ---
{
  "ts_utc": "2026-06-30T12:22:25.680580+00:00",
  "run_id": "theme_m1_20260630_175009",
  "event": "process_switched_out",
  "current_process": "C",
  "from": "C",
  "to": "B"
}
--- Pro

Chronological Process-Level GT Timeline

In [23]:
#Chronological process-level ground-truth timeline

process_timeline = []

for record in process_events:
    process_timeline.append({
        "timestamp": record.get("ts_utc"),
        "event": record.get("event"),
        "process": record.get("current_process"),
        "process_code": record.get("process_code"),
        "process_name": record.get("process_name"),
        "case_id": record.get("case_id"),
        "from": record.get("from"),
        "to": record.get("to")
    })

# Sort by timestamp
process_timeline = sorted(
    process_timeline,
    key=lambda x: x["timestamp"] if x["timestamp"] else ""
)

# Display the timeline
for i, record in enumerate(process_timeline, start=1):
    print(
        f"{i:02d} | "
        f"{record['timestamp']} | "
        f"{record['event']:<22} | "
        f"process={record['process']} | "
        f"case={record['case_id']} | "
        f"{record['from']} -> {record['to']}"
    )

01 | 2026-06-30T12:21:12.794418+00:00 | process_started        | process=H | case=BR-175009-001 | None -> None
02 | 2026-06-30T12:21:41.332221+00:00 | process_switched_out   | process=H | case=None | H -> C
03 | 2026-06-30T12:21:41.335282+00:00 | process_started        | process=C | case=LA-175009-001 | None -> None
04 | 2026-06-30T12:22:25.680580+00:00 | process_switched_out   | process=C | case=None | C -> B
05 | 2026-06-30T12:22:25.682643+00:00 | process_started        | process=B | case=PI-175009-001 | None -> None
06 | 2026-06-30T12:23:22.089303+00:00 | process_switched_out   | process=B | case=None | B -> A
07 | 2026-06-30T12:23:22.090864+00:00 | process_started        | process=A | case=RT-175009-001 | None -> None
08 | 2026-06-30T12:23:46.001092+00:00 | process_switched_out   | process=A | case=None | A -> M
09 | 2026-06-30T12:23:46.003666+00:00 | process_started        | process=M | case=SUP-175009-001 | None -> None
10 | 2026-06-30T12:24:20.921107+00:00 | process_switched_out

Analyze process starts and transitions

In [24]:
#Analyze process starts and process transitions

from collections import Counter

# Count process_started records by process
started_processes = [
    record["process"]
    for record in process_timeline
    if record["event"] == "process_started"
]

start_counts = Counter(started_processes)

print("Process start counts:")
for process, count in sorted(start_counts.items()):
    print(f"Process {process}: {count}")

print("\n" + "=" * 50)

# Analyze transitions from process_switched_out records
transitions = [
    (record["from"], record["to"])
    for record in process_timeline
    if record["event"] == "process_switched_out"
]

transition_counts = Counter(transitions)

print("Process transitions:")
for (from_process, to_process), count in sorted(transition_counts.items()):
    print(f"{from_process} -> {to_process}: {count}")

print("\n" + "=" * 50)

# Check whether a process_started event is immediately preceded
# by a switch to that same process.
print("Checking process starts after switches:\n")

for i, record in enumerate(process_timeline):
    if record["event"] != "process_started":
        continue

    if i > 0:
        previous = process_timeline[i - 1]

        if (
            previous["event"] == "process_switched_out"
            and previous["to"] == record["process"]
        ):
            relation = "STARTED AFTER SWITCH"
        else:
            relation = "STARTED WITHOUT IMMEDIATE SWITCH"
    else:
        relation = "FIRST PROCESS"

    print(
        f"{record['timestamp']} | "
        f"process={record['process']} | "
        f"case={record['case_id']} | "
        f"{relation}"
    )

Process start counts:
Process A: 4
Process B: 2
Process C: 4
Process E: 3
Process G: 4
Process H: 4
Process J: 3
Process M: 4
Process O: 3

Process transitions:
A -> C: 1
A -> E: 1
A -> M: 2
B -> A: 2
C -> B: 3
C -> E: 1
E -> C: 1
E -> M: 1
E -> O: 1
G -> A: 1
G -> J: 2
H -> A: 1
H -> C: 1
H -> J: 1
J -> G: 2
J -> O: 1
M -> C: 1
M -> G: 1
M -> H: 1
O -> E: 1

Checking process starts after switches:

2026-06-30T12:21:12.794418+00:00 | process=H | case=BR-175009-001 | FIRST PROCESS
2026-06-30T12:21:41.335282+00:00 | process=C | case=LA-175009-001 | STARTED AFTER SWITCH
2026-06-30T12:22:25.682643+00:00 | process=B | case=PI-175009-001 | STARTED AFTER SWITCH
2026-06-30T12:23:22.090864+00:00 | process=A | case=RT-175009-001 | STARTED AFTER SWITCH
2026-06-30T12:23:46.003666+00:00 | process=M | case=SUP-175009-001 | STARTED AFTER SWITCH
2026-06-30T12:24:20.924171+00:00 | process=C | case=LA-175009-002 | STARTED AFTER SWITCH
2026-06-30T12:24:53.808438+00:00 | process=B | case=PI-175009-002 | S

Compare task_started and process_started records

In [25]:
#Compare task_started and process_started records

task_events = []

with open(gt_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        if record.get("event") == "task_started":
            task_events.append(record)

print("Number of task_started records:", len(task_events))

print("\nTask-start records:\n")

for i, record in enumerate(task_events, start=1):
    print(
        f"{i:02d} | "
        f"{record.get('ts_utc')} | "
        f"process={record.get('current_process')} | "
        f"case={record.get('case_id')} | "
        f"task={record.get('task')} | "
        f"name={record.get('task_name')}"
    )

Number of task_started records: 32

Task-start records:

01 | 2026-06-30T12:21:12.797484+00:00 | process=H | case=None | task=None | name=None
02 | 2026-06-30T12:21:41.338448+00:00 | process=C | case=None | task=None | name=None
03 | 2026-06-30T12:22:25.684775+00:00 | process=B | case=None | task=None | name=None
04 | 2026-06-30T12:23:22.093472+00:00 | process=A | case=None | task=None | name=None
05 | 2026-06-30T12:23:46.005209+00:00 | process=M | case=None | task=None | name=None
06 | 2026-06-30T12:24:20.925692+00:00 | process=C | case=None | task=None | name=None
07 | 2026-06-30T12:24:53.811496+00:00 | process=B | case=None | task=None | name=None
08 | 2026-06-30T12:26:02.532525+00:00 | process=H | case=None | task=None | name=None
09 | 2026-06-30T12:26:32.491258+00:00 | process=A | case=None | task=None | name=None
10 | 2026-06-30T12:26:55.502710+00:00 | process=M | case=None | task=None | name=None
11 | 2026-06-30T12:27:38.181381+00:00 | process=H | case=None | task=None | name=No

Measure GT timing gaps

In [27]:
# Cell 14: Match each process_started with the nearest task_started

from datetime import datetime

def parse_timestamp(ts):
    return datetime.fromisoformat(ts)

# Sort both event types chronologically
process_starts = sorted(
    [
        record for record in process_timeline
        if record["event"] == "process_started"
    ],
    key=lambda x: x["timestamp"]
)

task_starts_sorted = sorted(
    task_events,
    key=lambda x: x["ts_utc"]
)

print("Process start → nearest task start gaps:\n")

for process_record in process_starts:

    process_time = parse_timestamp(process_record["timestamp"])

    # Find task_started events occurring at or after the process start
    candidate_tasks = [
        task for task in task_starts_sorted
        if parse_timestamp(task["ts_utc"]) >= process_time
    ]

    if candidate_tasks:

        nearest_task = min(
            candidate_tasks,
            key=lambda task:
                parse_timestamp(task["ts_utc"]) - process_time
        )

        task_time = parse_timestamp(nearest_task["ts_utc"])

        gap_ms = (task_time - process_time).total_seconds() * 1000

        print(
            f"{process_record['process']} | "
            f"{process_record['case_id']} | "
            f"gap = {gap_ms:.3f} ms"
        )

    else:
        print(
            f"{process_record['process']} | "
            f"{process_record['case_id']} | "
            f"No matching task_started found"
        )

Process start → nearest task start gaps:

H | BR-175009-001 | gap = 3.066 ms
C | LA-175009-001 | gap = 3.166 ms
B | PI-175009-001 | gap = 2.132 ms
A | RT-175009-001 | gap = 2.608 ms
M | SUP-175009-001 | gap = 1.543 ms
C | LA-175009-002 | gap = 1.521 ms
B | PI-175009-002 | gap = 3.058 ms
H | BR-175009-002 | gap = 3.076 ms
A | RT-175009-002 | gap = 3.056 ms
M | SUP-175009-002 | gap = 3.448 ms
H | BR-175009-003 | gap = 1.529 ms
H | BR-175009-004 | gap = 1.541 ms
J | PM-175009-001 | gap = 3.067 ms
G | EXP-175009-001 | gap = 2.535 ms
G | EXP-175009-002 | gap = 2.589 ms
A | RT-175009-003 | gap = 1.547 ms
C | LA-175009-003 | gap = 2.713 ms
A | RT-175009-004 | gap = 1.083 ms
E | OB-175009-001 | gap = 3.119 ms
M | SUP-175009-003 | gap = 1.537 ms
M | SUP-175009-004 | gap = 2.524 ms
G | EXP-175009-003 | gap = 2.047 ms
J | PM-175009-002 | gap = 2.025 ms
G | EXP-175009-004 | gap = 2.391 ms
J | PM-175009-003 | gap = 2.521 ms
O | RET-175009-001 | gap = 1.516 ms
E | OB-175009-002 | gap = 2.592 ms
C | 

Load the complete event stream for this session

In [29]:
# Load and reconstruct the complete event stream for the session

all_events = []

# Find every events.jsonl file inside the session
event_files = sorted(session.rglob("events.jsonl"))

print("Number of event files:", len(event_files))

# Load events from every event file
for event_file in event_files:

    with open(event_file, "r", encoding="utf-8") as f:

        for line in f:
            line = line.strip()

            if not line:
                continue

            event = json.loads(line)

            # Store the source file for traceability
            event["_source_file"] = str(event_file)

            all_events.append(event)

print("Total raw events loaded:", len(all_events))


# Sort the complete event stream chronologically
all_events = sorted(
    all_events,
    key=lambda x: x.get("timestamp_ms", 0)
)


# Display the first 10 events safely
print("\nFirst 10 events chronologically:\n")

for event in all_events[:10]:

    # Some events may have context=None
    context = event.get("context") or {}

    # Some contexts may not contain active_app
    active_app = context.get("active_app") or {}

    print(
        event.get("timestamp_iso"),
        "|",
        event.get("event_type"),
        "|",
        active_app.get("app_name")
    )

Number of event files: 2
Total raw events loaded: 2348

First 10 events chronologically:

2026-06-30T12:19:53.821Z | session_start | None
2026-06-30T12:19:54.435Z | mouse_click | WindowsTerminal
2026-06-30T12:19:54.465Z | app_switch | WindowsTerminal
2026-06-30T12:19:54.524Z | screenshot_smart | WindowsTerminal
2026-06-30T12:19:55.403Z | keystroke | WindowsTerminal
2026-06-30T12:19:55.466Z | screenshot_smart | WindowsTerminal
2026-06-30T12:19:55.514Z | keystroke | WindowsTerminal
2026-06-30T12:19:57.385Z | keystroke | WindowsTerminal
2026-06-30T12:19:57.411Z | screenshot_smart | WindowsTerminal
2026-06-30T12:19:57.520Z | keystroke | WindowsTerminal


GT process starts to the raw event stream

In [30]:
# Inspect raw events around process starts

print("Raw events around each process start:\n")

for process_record in process_starts:

    process_time = parse_timestamp(process_record["timestamp"])

    print("\n" + "=" * 80)
    print(
        f"PROCESS START: {process_record['process']} | "
        f"Case: {process_record['case_id']} | "
        f"Time: {process_record['timestamp']}"
    )
    print("=" * 80)

    # Show events occurring within 5 seconds before and after process start
    for event in all_events:

        event_time = parse_timestamp(event["timestamp_iso"])

        time_diff = (event_time - process_time).total_seconds()

        if -5 <= time_diff <= 5:

            context = event.get("context") or {}
            active_app = context.get("active_app") or {}

            print(
                f"{time_diff:+7.3f}s | "
                f"{event.get('event_type'):<20} | "
                f"app={active_app.get('app_name')}"
            )

Raw events around each process start:


PROCESS START: H | Case: BR-175009-001 | Time: 2026-06-30T12:21:12.794418+00:00
 -0.391s | app_switch           | app=Microsoft Excel
 -0.253s | screenshot_smart     | app=Microsoft Excel
 +0.176s | app_switch           | app=Microsoft Excel
 +0.273s | app_switch           | app=Microsoft Excel
 +0.298s | screenshot_smart     | app=Microsoft Excel
 +0.326s | app_switch           | app=Microsoft Excel
 +0.384s | app_switch           | app=Microsoft Excel
 +0.438s | app_switch           | app=Microsoft Excel
 +0.506s | app_switch           | app=Microsoft Excel
 +0.551s | app_switch           | app=Microsoft Excel
 +0.625s | app_switch           | app=Microsoft Excel
 +0.661s | app_switch           | app=Microsoft Excel
 +0.715s | app_switch           | app=Microsoft Excel
 +0.770s | app_switch           | app=Microsoft Excel
 +0.840s | screenshot_smart     | app=Microsoft Excel
 +2.834s | app_switch           | app=Microsoft Excel
 +2.878s | app_s

Summarize raw activity around process starts

In [31]:
#  Summarize raw events around each process start

from collections import Counter

print("Summary of raw activity around process starts\n")

for process_record in process_starts:

    process_time = parse_timestamp(process_record["timestamp"])

    nearby_events = []

    for event in all_events:

        event_time = parse_timestamp(event["timestamp_iso"])
        time_diff = (event_time - process_time).total_seconds()

        # 2 seconds before to 2 seconds after
        if -2 <= time_diff <= 2:
            nearby_events.append(event)

    event_types = Counter(
        event.get("event_type")
        for event in nearby_events
    )

    apps = Counter()

    for event in nearby_events:

        context = event.get("context") or {}
        active_app = context.get("active_app") or {}
        app_name = active_app.get("app_name")

        if app_name:
            apps[app_name] += 1

    print(
        f"{process_record['process']} | "
        f"{process_record['case_id']} | "
        f"{process_record['timestamp']}"
    )

    print(
        f"  Events in ±2 sec: {len(nearby_events)}"
    )

    print(
        f"  Event types: {dict(event_types)}"
    )

    print(
        f"  Top apps: {apps.most_common(5)}"
    )

    print()

Summary of raw activity around process starts

H | BR-175009-001 | 2026-06-30T12:21:12.794418+00:00
  Events in ±2 sec: 15
  Event types: {'app_switch': 12, 'screenshot_smart': 3}
  Top apps: [('Microsoft Excel', 15)]

C | LA-175009-001 | 2026-06-30T12:21:41.335282+00:00
  Events in ±2 sec: 17
  Event types: {'app_switch': 14, 'screenshot_smart': 3}
  Top apps: [('Google Chrome', 17)]

B | PI-175009-001 | 2026-06-30T12:22:25.682643+00:00
  Events in ±2 sec: 0
  Event types: {}
  Top apps: []

A | RT-175009-001 | 2026-06-30T12:23:22.090864+00:00
  Events in ±2 sec: 0
  Event types: {}
  Top apps: []

M | SUP-175009-001 | 2026-06-30T12:23:46.003666+00:00
  Events in ±2 sec: 17
  Event types: {'app_switch': 14, 'screenshot_smart': 3}
  Top apps: [('Google Chrome', 17)]

C | LA-175009-002 | 2026-06-30T12:24:20.924171+00:00
  Events in ±2 sec: 14
  Event types: {'app_switch': 12, 'screenshot_smart': 2}
  Top apps: [('Google Chrome', 14)]

B | PI-175009-002 | 2026-06-30T12:24:53.808438+00:00

Find nearest raw event to each process start

In [32]:
# Find the nearest raw event to each process start

print("Nearest raw event to each process start:\n")

for process_record in process_starts:

    process_time = parse_timestamp(process_record["timestamp"])

    # Find the raw event with the smallest absolute time difference
    nearest_event = min(
        all_events,
        key=lambda event: abs(
            (
                parse_timestamp(event["timestamp_iso"])
                - process_time
            ).total_seconds()
        )
    )

    nearest_time = parse_timestamp(nearest_event["timestamp_iso"])

    gap_ms = (
        nearest_time - process_time
    ).total_seconds() * 1000

    context = nearest_event.get("context") or {}
    active_app = context.get("active_app") or {}

    print(
        f"{process_record['process']} | "
        f"{process_record['case_id']}"
    )

    print(
        f"  GT time:       {process_record['timestamp']}"
    )

    print(
        f"  Nearest event: {nearest_event.get('timestamp_iso')}"
    )

    print(
        f"  Event type:    {nearest_event.get('event_type')}"
    )

    print(
        f"  App:           {active_app.get('app_name')}"
    )

    print(
        f"  Gap:           {gap_ms:.3f} ms"
    )

    print()

Nearest raw event to each process start:

H | BR-175009-001
  GT time:       2026-06-30T12:21:12.794418+00:00
  Nearest event: 2026-06-30T12:21:12.970Z
  Event type:    app_switch
  App:           Microsoft Excel
  Gap:           175.582 ms

C | LA-175009-001
  GT time:       2026-06-30T12:21:41.335282+00:00
  Nearest event: 2026-06-30T12:21:41.508Z
  Event type:    app_switch
  App:           Google Chrome
  Gap:           172.718 ms

B | PI-175009-001
  GT time:       2026-06-30T12:22:25.682643+00:00
  Nearest event: 2026-06-30T12:22:22.414Z
  Event type:    mouse_click
  App:           Google Chrome
  Gap:           -3268.643 ms

A | RT-175009-001
  GT time:       2026-06-30T12:23:22.090864+00:00
  Nearest event: 2026-06-30T12:23:25.385Z
  Event type:    mouse_click
  App:           Google Chrome
  Gap:           3294.136 ms

M | SUP-175009-001
  GT time:       2026-06-30T12:23:46.003666+00:00
  Nearest event: 2026-06-30T12:23:46.174Z
  Event type:    app_switch
  App:           Goo

# Dataset A — Initial Audit Summary

## What I Understand So Far

### Dataset Structure
- Dataset A contains **63 sessions**.
- Each session contains:
  - multiple `chunks/`
  - `gt.jsonl`
  - `gt_manifest.json`
- Chunks contain the raw computer activity in `events.jsonl`.
- A session can be split across multiple chunks, so chunks should be combined when reconstructing the session.

### Raw Event Data
For the first session inspected:
- **2 `events.jsonl` files**
- **2,348 total events**
- Events include:
  - mouse clicks
  - keystrokes
  - application switches
  - screenshots
  - session events
- Events contain timestamps, application context, payload, UI information, and other metadata.
- Events should be sorted by timestamp rather than relying on JSONL line order.
- Some nested fields can be missing/`None`, so the code needs to handle them safely.

### Ground Truth
`gt.jsonl` contains different types of records, not only process boundaries.

For the first session:
- `process_started` → **31**
- `process_switched_out` → **26**
- `process_suspended` → **1**
- `process_resumed` → **1**
- `task_started` → **32**
- Total GT records → **175**

### Process Behavior
- There are **9 process types**: `A, B, C, E, G, H, J, M, O`.
- The same process type can occur multiple times with different **case IDs**.
- Processes can switch/interleave.
- A `process_switched_out` does **not necessarily mean the process is completed**.
- Some `process_started` events occur without an immediately preceding process switch.
- Therefore, process segmentation is more complicated than simply detecting application switches.

### `task_started`
- There are **32 `task_started`** records and **31 `process_started`** records.
- In the inspected session, `task_started` occurs approximately **1–3 ms after process_started**.
- It appears to be closely associated with process initiation, but its exact independent meaning is not yet established.

### Raw Events vs Ground Truth
- Raw events do not contain an obvious event saying "this business process has started."
- Some process starts have many raw events close to the GT timestamp.
- Other process starts have no events within a ±2-second window.
- The nearest raw event can be several seconds away.
- Therefore, a simple fixed time window or `app_switch = process boundary` rule is unlikely to be sufficient.

## Main Insight

**The challenge is to infer business-process segments from low-level computer events, while accounting for process repetition, interleaving, noise, and imperfect temporal alignment with the ground truth.**

## Next Step

Before designing the segmentation algorithm:

**Check whether these observations from the first session also hold across all 63 Dataset A sessions.**